# BlurBall — Passo 1 no **Kaggle** (vídeo completo)

Corre o BlurBall pré-treinado no `Parada4.mp4` e mede o recall da bola vs os ~72% do YOLO.
Vantagem do Kaggle: a sessão é estável e não se apaga a meio como o Colab grátis.

## Antes de correr (fazer 1 vez, no painel à direita)
1. **Session options → Accelerator:** escolhe **GPU T4 x2** (ou P100).
2. **Session options → Internet:** liga **ON** (precisa de conta verificada por telemóvel).
3. **Input → + Add Input → Datasets:** adiciona o dataset onde puseste o `Parada4.mp4`.
   (Se ainda não criaste: *Create → New Dataset → Upload* o Parada4.mp4 da tua pasta do projeto.)

Depois corre as células por ordem, de cima para baixo.


### 1) Confirmar GPU

In [ ]:
!nvidia-smi

### 2) Clonar o BlurBall e instalar dependências (~3 min)

In [ ]:
import os
if not os.path.isdir('/kaggle/working/blurball'):
    !git clone -q https://github.com/cogsys-tuebingen/blurball.git /kaggle/working/blurball
!pip install -q -r /kaggle/working/blurball/requirements.txt
print("deps ok")

### 3) Descarregar os pesos pré-treinados
Os ficheiros não têm extensão (`blurball_best`, ...), por isso apanhamo-los pelo tamanho.

In [ ]:
import glob, os
os.makedirs('/kaggle/working/weights', exist_ok=True)
if not [p for p in glob.glob('/kaggle/working/weights/**/*', recursive=True) if os.path.isfile(p)]:
    !wget -q -O /kaggle/working/weights.zip "https://cloud.cs.uni-tuebingen.de/index.php/s/6Z8TpM3sXRKHzGC/download"
    !cd /kaggle/working/weights && unzip -o -q /kaggle/working/weights.zip
pesos = [p for p in glob.glob('/kaggle/working/weights/**/*', recursive=True)
         if os.path.isfile(p) and os.path.getsize(p) > 1_000_000]
BLURBALL_W = [p for p in pesos if 'blurball' in os.path.basename(p).lower()][0]
print(">>> BLURBALL_W =", BLURBALL_W)

### 4) Apontar o repo para esta pasta (WASB_ROOT)

In [ ]:
!sed -i 's|^WASB_ROOT:.*|WASB_ROOT: /kaggle/working/blurball|' /kaggle/working/blurball/src/configs/global.yaml
!cat /kaggle/working/blurball/src/configs/global.yaml

### 5) Localizar o vídeo (do dataset) e copiá-lo para a pasta de trabalho
O `/kaggle/input` é só de leitura; o BlurBall precisa de escrever ao lado do vídeo, por isso copiamos.

In [ ]:
import glob, shutil, os
src = glob.glob('/kaggle/input/**/Parada4*.mp4', recursive=True)
assert src, "Nao encontrei Parada4.mp4 no Input — adicionaste o dataset? (painel direito -> Add Input)"
VIDEO_PATH = '/kaggle/working/Parada4.mp4'
shutil.copy(src[0], VIDEO_PATH)
print("Video pronto:", VIDEO_PATH, os.path.getsize(VIDEO_PATH)//1024//1024, "MB")

# OPCIONAL: para um teste rápido só nos primeiros 120s (4 rallies), descomenta:
# !ffmpeg -y -loglevel error -i /kaggle/working/Parada4.mp4 -t 120 -c copy /kaggle/working/Parada4_120s.mp4
# VIDEO_PATH = '/kaggle/working/Parada4_120s.mp4'

### 6) Correr o BlurBall (step=1, threshold=0.7)
Vídeo completo = ~15–20 min. No Kaggle podes deixar correr; a sessão aguenta.

In [ ]:
# THRESHOLD 0.7 -> 0.4
# A 0.7 so 46% dos frames tem bola. A 0.4 sobem para ~76%.
# Ganham-se as detecoes DIFICEIS: bola borrada, rapida, pequena ao fundo
# -- os momentos que mais interessam (servico, remate).
# O lixo extra e limpo depois: IMOVEIS + CONTINUIDADE.

import os; os.environ['HYDRA_FULL_ERROR']='1'
%cd /kaggle/working/blurball/src
!python main.py --config-name=inference_blurball \
    detector.model_path="{BLURBALL_W}" +input_vid="{VIDEO_PATH}" \
    detector.step=1 detector.postprocessor.score_threshold=0.4 runner.vis_hm=false

### 7) Localizar os resultados

In [ ]:
import os
vname = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
TRAJ = f'/kaggle/working/frames_{vname}/traj.csv'
OVERLAY = '/kaggle/working/frames.mp4'
print("traj.csv:", TRAJ, "->", os.path.exists(TRAJ))
print("overlay :", OVERLAY, "->", os.path.exists(OVERLAY))

### 8) O número — deteção da bola dentro dos rallies (recall proxy) vs 72%
Dentro dos 12 rallies reais a bola está sempre em jogo, por isso a % de frames com bola detetada
é comparável ao recall do YOLO (72%).

In [ ]:
import pandas as pd, cv2, numpy as np
df = pd.read_csv(TRAJ); vis = df['Visibility'].values; N = len(df)
fps = cv2.VideoCapture(VIDEO_PATH).get(cv2.CAP_PROP_FPS) or 29.97

# 12 rallies reais (segundos) — ground_truth_parada4. Se correste o clip de 120s, so' os 1-4 contam.
GT = [(1,38.0,41.5),(2,46.8,67.5),(3,77.6,85.5),(4,95.9,111.1),(5,122.4,135.9),(6,157.9,169.4),
      (7,178.1,186.5),(8,197.0,202.1),(9,210.5,216.3),(10,229.9,237.3),(11,249.6,255.0),(12,263.8,276.4)]
inplay = np.zeros(N, bool)
print("Frames:", N, "| fps=%.2f" % fps, "| deteção global: %.1f%%" % (vis.mean()*100))
print("Por rally:")
for k,a,b in GT:
    f0,f1 = int(a*fps), int(b*fps)
    if f0 >= N: continue
    f1 = min(N-1, f1); inplay[f0:f1+1] = True
    print(f"  rally {k:>2} ({a:>5}-{b:>5}s): {vis[f0:f1+1].mean()*100:5.1f}%")
print("\n>>> Deteção DENTRO dos rallies: %.1f%%   (YOLO atual = 72%%)" % (vis[inplay].mean()*100))

### (opcional) Guardar os resultados
Ficam em `/kaggle/working/` e podes descarregá-los pelo painel **Output** (Data → /kaggle/working).
O `traj.csv` é o `ball_xy` que depois liga ao `rallies_bola.py` (passo 3), sem mudar as regras v9.

## Guardar com o nome certo

O ficheiro tem de se chamar **`traj_frames_Parada4_thr04.csv`**.
No Kaggle, os outputs ficam em `/kaggle/working/` — descarrega dali pelo painel da direita.


In [ ]:
import glob, shutil, os, csv

cands = glob.glob('/kaggle/working/**/traj*.csv', recursive=True)
cands += glob.glob('**/traj*.csv', recursive=True)
cands = sorted(set(cands), key=os.path.getmtime, reverse=True)
assert cands, 'Nao encontrei nenhum traj.csv'
src = cands[0]
print('encontrado:', src)

DST = '/kaggle/working/traj_frames_Parada4_thr04.csv'
shutil.copy(src, DST)

rows = list(csv.DictReader(open(DST)))
vis = sum(1 for r in rows if r.get('Visibility') == '1')
print(f'\nframes: {len(rows)}')
print(f'com bola: {vis}  ({100*vis/len(rows):.1f}%)   <-- a thr=0.7 dava 46.2%')
print('\n>>> Descarrega o ficheiro no painel da DIREITA (Output) e poe em dados_parada4/')


## ⭐ GUARDAR OS PESOS (importante — fazer uma vez)

Os pesos do BlurBall vêm de um **link partilhado de um servidor da Universidade de Tübingen**.
Não é a Drive de um colega: é um link público de um grupo de investigação, que pode mudar,
expirar ou desaparecer **sem aviso e sem ninguém a quem pedir**.

E é o modelo que **substituiu o YOLO** (recall 85,6% vs 67%) e sustenta metade do M1.
**Se aquele link morrer, não há como o refazer** — os pesos são do paper deles, treinados
com dados que não temos.

Esta célula copia-os para `/kaggle/working/pesos_blurball/`.
**Descarrega-os no painel Output (à direita) e guarda no teu Drive + no Mac.**


In [ ]:
import glob, os, shutil

DST = '/kaggle/working/pesos_blurball'
os.makedirs(DST, exist_ok=True)

# os pesos NAO tem extensao -> apanhar pelos ficheiros grandes (>1 MB)
achados = [p for p in glob.glob('/kaggle/working/**/*', recursive=True)
           if os.path.isfile(p) and os.path.getsize(p) > 1_000_000
           and 'pesos_blurball' not in p
           and not p.endswith(('.mp4', '.csv', '.ipynb'))]

print('Pesos encontrados:')
for src in sorted(set(achados)):
    nome = os.path.basename(src)
    shutil.copy(src, os.path.join(DST, nome))
    print(f'   {nome:30s} {os.path.getsize(src)/1e6:7.1f} MB  -> copiado')

if not achados:
    print('   NENHUM. Corre primeiro as celulas de cima (descarga dos pesos).')
else:
    print(f'\n>>> Estao em {DST}')
    print('>>> DESCARREGA-OS no painel OUTPUT (direita) e guarda na Drive + no Mac.')
    print('>>> E a unica peca do projeto que e VERDADEIRAMENTE insubstituivel.')
